# Week 7 — CTEs and Advanced Analytics: Window Functions
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Today's demo gave you three new columns to put inside a named step: `LAG()` for looking one row
back, `RANK()` for a real position column, and `NTILE()` for equal-sized buckets. These five
questions put `RANK()` and the CTE pipeline to work on numbers you can check — every expected value
below is a verified Olist fact, so a ✅ means your query is right, not just plausible.

Each question comes as **three cells**:

1. A **question** with the task and an **Expected** value.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the
   result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the
   cell then displays the rows your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom.

Three reminders that will save you time. A CTE lives for exactly **one** statement, so every answer
cell must carry its own complete `WITH` block. The check cells read specific **column aliases**, so
name your columns exactly as each question asks. And the big one from today's demo: **`WHERE` runs
before window functions**, so you can never filter on a `RANK()` column in the same `SELECT` that
creates it — compute it in a CTE, then filter outside.

### Setup — run this cell first

Loads the 8 Olist tables into a file-based SQLite database and connects the `%%sql` magic to it.
Because `autopandas` is on, every `%%sql` result comes back as a pandas DataFrame, which is what
lets the check cells inspect your answer.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Question 1 — Which year was Olist's biggest?

Olist's board wants the yearly order trend, ranked, on one slide.

Write a query with a CTE named `yearly_orders` that reads `orders`, groups by
`strftime('%Y', order_purchase_timestamp) AS year`, and produces one row per year with
`COUNT(*) AS order_count`.

In the **outer** query, select `year`, `order_count`, and add a real position column:

```sql
RANK() OVER (ORDER BY order_count DESC) AS rank_num
```

Return **all** the years — no filter — sorted with `ORDER BY order_count DESC` so the biggest year
lands on the first row. Leaving the rows in means the rank is computed over the whole set, which is
the only way it means anything.

**Expected:** 3 rows; the top row is 2018 with an `order_count` of 54,011 and a `rank_num` of 1

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert q1.shape[0] == 3, "Q1: expected 3 rows, one per year (2016, 2017, 2018)"
assert str(q1.iloc[0]['year']) == '2018', "Q1: expected 2018 on the first row — sort by order_count DESC"
assert int(q1.iloc[0]['order_count']) == 54011, "Q1: expected 54,011 orders for 2018"
assert int(q1.iloc[0]['rank_num']) == 1, "Q1: expected the top row's rank_num to be 1"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Finding Black Friday in the data

The demo's `LAG()` query showed you a spike around November 2017 without ever printing the number
behind it. Go and get the number.

Write a query with a CTE named `monthly_orders` that reads `orders`, groups by
`strftime('%Y-%m', order_purchase_timestamp) AS month`, and produces one row per month with
`COUNT(*) AS order_count`. In the **outer** query, read from `monthly_orders` and return just the
row for `'2017-11'`.

Note the format string: `'%Y-%m'` gives you `2017-11`, while `'%Y'` gives you `2017`. Ask for the
wrong one and you will get a perfectly valid answer to a different question.

**Expected:** one row — 2017-11 with an `order_count` of 7,544 (the single biggest month in the
dataset, and the reason the demo's growth column swings so hard in December)

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert int(q2.iloc[0]['order_count']) == 7544, "Q2: expected 7,544 orders in 2017-11"
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — The single biggest seller on the platform

Now the pattern the demo's mini-challenge was really about: **compute the rank inside, filter
outside.**

Build a three-step query:

1. `seller_revenue` — from `order_items`, one row per `seller_id` with
   `ROUND(SUM(price), 2) AS total_revenue`.
2. `ranked` — reads **from `seller_revenue`** and adds
   `RANK() OVER (ORDER BY total_revenue DESC) AS rank_num`.
3. The outer query reads from `ranked` and returns only the row where `rank_num = 1`.

Do **not** try to write `WHERE rank_num = 1` next to the `RANK()` itself. `WHERE` runs before window
functions, so SQLite will either tell you `no such column: rank_num` or — worse, if you filter on
something else — hand you a rank computed over a single surviving row. That second version runs
without error, which is exactly what makes it dangerous.

**Expected:** one row — the top seller, with a `total_revenue` of 229,472.63

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert round(float(q3.iloc[0]['total_revenue']), 2) == 229472.63, \
    "Q3: expected total_revenue of 229,472.63 for the rank 1 seller"
assert int(q3.iloc[0]['rank_num']) == 1, "Q3: expected the returned row to carry rank_num = 1"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — The top category, pulled out by rank

Same shape as Question 3, but on categories — and this time the CTE has joins in it.

1. `category_revenue` — join `order_items` to `products` on `product_id`, and `products` to
   `product_category_translation` on `product_category_name`. Group by
   `t.product_category_name_english AS category` and produce `ROUND(SUM(oi.price), 2) AS revenue`.
2. `ranked` — reads **from `category_revenue`** and adds
   `RANK() OVER (ORDER BY revenue DESC) AS rank_num`.
3. The outer query returns only the row where `rank_num = 1`.

One grain check before you run it: `order_items` is the only many-rows-per-order table in this
query, so `SUM(oi.price)` is at the right grain. Bring `order_payments` or `order_reviews` into the
same join and the revenue would inflate — that is the fan-out trap from the demo.

**Expected:** one row — health_beauty, with a `revenue` of 1,258,681.34

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert round(float(q4.iloc[0]['revenue']), 2) == 1258681.34, \
    "Q4: expected revenue of 1,258,681.34 for the rank 1 category"
assert str(q4.iloc[0]['category']) == 'health_beauty', "Q4: expected health_beauty as the rank 1 category"
print("✅ Q4 correct")
q4  # show the result of your query

## Question 5 — How many customers gave five stars?

Last one, and the CTE here is doing the simplest job a CTE can do: naming a step so the outer query
reads like a sentence.

Write a query with a CTE named `score_counts` that reads `order_reviews`, groups by `review_score`,
and produces `COUNT(*) AS n`. In the outer query, read from `score_counts` and return only the row
where `review_score = 5`.

`review_score` is a number, not text — so it is `= 5`, not `= '5'`.

**Expected:** one row — `review_score` 5 with an `n` of 57,328 (57.8% of all 99,224 reviews, which
is worth holding in your head next time someone tells you a marketplace's ratings are evenly
spread)

In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert int(q5.iloc[0]['n']) == 57328, "Q5: expected 57,328 five-star reviews"
print("✅ Q5 correct")
q5  # show the result of your query